Etapa 1 - EDA + Baselines + MLflow Tracking

## Objetivos

- Carregar e explorar os dataset TELCO CUSTOMER CHURN (IBM)
- Realizar EDA completa: volume, qualidade, distrubição, data readiness
- Tratar missing values e anomalias
- Definir métricas técnicas e de negócio
- Treinar baselines (DummyClassifier + Regressão Logística)
- Registrar experimentos no MLflow

## Referências

- Ciclo de Vida, Aula 01 (Business Understanding + Data Understanding)
- Fundamentos, Aulas 01-02 (Baselines)
- Fundamentos, Aula 05 (Métricas)
- Ciclo de Vida, Aula 02 (MLflow tracking)

##### 1. Configuração do ambiente

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Visualização

import matplotlib.pyplot as mtl
import seaborn as sns
sns.set_style("whitegrid")
mtl.rcParams["figure.figsize"] = (12, 6)

# estatistica
from scipy import stats

# Machine Learning (Scikit-Learn é tipo o Spring do ML...framwork padrao)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)

# MLflow eh um tracking de experimentos (como um Micrometer/Prometheus para ML)
import mlflow
import mlflow.sklearn

# Reprodutibilidade seed fixo
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

skitlearn_version = __import__("sklearn").__version__
mlflow_version = __import__("mlflow").__version__

print("Successfully configured environment")
print(f"pandas version: {pd.__version__}")
print(f"scikit-learn version: {skitlearn_version}")
print(f"mlflow version: {mlflow_version}")

## 2. Carregamento dos Dados

#### Sobre o dataset

**Telco Customer Churn (IBM)** Dataset publico de uma operadora de telecomunicações ficticia.

- **Registros** 7.043 clientes
- **Feature** 21 colunas (demográficas, de conta e de serviços)
- **Target** `Churn` (yes/no) - se o cliente cancelou o serviço
- **Fonte** [IBM Sample Datasets / Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

#### Como baixar
Execute a célula abaixo. Ela tenta baixar direto do GitHub da IBM.
Se não funcionar, baixe manualmente do [Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) e salve como `data/raw/telco_customer_churn.csv`.

In [ ]:
import os
from pathlib import Path

# Configura o caminho do dataset
RAW_DATA_DIR = Path("../data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH = RAW_DATA_DIR / "telco_customer_churn.csv"

if not DATASET_PATH.exists():
    print("The dataset was downloaded from the IBM repository on GitHub.")
    url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
    try:
        data_frame = pd.read_csv(url)
        data_frame.to_csv(DATASET_PATH, index=False)
        print(f"Dataset salvo em: {DATASET_PATH}")
    except Exception as e:
        print(f"Error downloading the dataset: {e}")
        print("\nDownload manually from Kaggle:")
        print("https://www.kaggle.com/datasets/blastchar/telco-customer-churn")
        print(f"Save as: {DATASET_PATH}")
else:
    print(f"Dataset já existe em: {DATASET_PATH}")
    pd.read_csv(DATASET_PATH)

print(f"\nShape dos dados: {data_frame.shape}")
print(f"\nPrimeiras linhas:")

### 3. EDA - Análise Exploratória de Dados

##### 3.1 Visão Geral do Dataset

- `df.info()` -> mostra tipos de cada coluna e quantos valores nãonulos (como metadata de uma tabela)
- `df.describe()` -> estatísticas descritivas (min, max, média, mediana, quartis)

In [ ]:
print("=" * 60)
print("GENERAL INFORMATION ABOUT THE DATASET")
print("=" * 60)
print(f"\nDimensions: {data_frame.shape[0]} lines x {data_frame.shape[1]} columns")
print(f"Memory used: {data_frame.memory_usage(deep=True).sum() / 1024:.1f} Kb")
print()
print(data_frame.info)

In [ ]:
print("=" * 60)
print("Descriptive Statistics: Numerical")
print("=" * 60)
data_frame.describe()

In [ ]:
print("=" * 60)
print("Descriptive statistics: categorical")
print("=" * 60)
data_frame.describe(include="object")

#### 3.2 Analise de Missing Values

Verificar se há dados faltantes do dataset Telco. A coluna `TotalCharges` pode ter valores em branco, ou espaços que o Pandas não detecta autmaticamente como `NaN`, precisam ser tratados.

In [ ]:
print("=" * 60)
print("MISSING VALUES ANALYSIS")
print("=" * 60)

# TotalCharges pode conter espaços em branco que não são NaN
# Vamos converter para numérico forçando erros para NaN
data_frame["TotalCharges"] = pd.to_numeric(data_frame["TotalCharges"], errors="coerce")

# aqui verifica os missing values
missing_values = pd.DataFrame({
    "column": data_frame.columns,
    "missing_count": data_frame.isnull().sum(),
    "missing_percentage (%)": (data_frame.isnull().sum() / len(data_frame) * 100).round(2)
})
missing_values = missing_values[missing_values["missing_count"] > 0].sort_values("missing_percentage (%)", ascending=False)
print(missing_values)

if len(missing_values) > 0:
    print(missing_values.to_string(index=False))
    print(f"\nColumns with missing values: {data_frame.isnull().any(axis=1).sum()}")

    #visualização dos missing values
    mtl.figure(figsize=(10, 4))
    mtl.barh(missing_values["column"], missing_values["Percentage (%)"], color="coral")
    mtl.xlabel("percentage of missing values (%)")
    mtl.title("Missing Values Analysis")
    mtl.tight_layout
    mtl.show()
else:
    print("No missing values found in the dataset.")

In [ ]:
# Tratar missing values em TotalCharges
# Clientes com tenure=0 (recém-chegados) têm TotalCharges vazio
# Estratégia: preencher com 0 (faz sentido: se acabou de chegar, não pagou nada)
print ("Customers with zero TotalCharges:")
print (data_frame[data_frame["TotalCharges"].isnull()][["customerID", "tenure", "MonthlyCharges", "TotalCharges"]])

data_frame["TotalCharges"].fillna(0, inplace=True)
#data_frame["TotalCharges"] = data_frame["TotalCharges"].fillna(0)

print(f"\nMissing values ​​after treatment: {data_frame.isnull().sum().sum()}")

#data_frame.isnull()	matriz de True/False
#data_frame.isnull().sum()	nulos por coluna
#data_frame.isnull().sum().sum()	total geral de nulos

#### 3.3 Análise da Variável Target (Churn)

Precisamos entender o balanceamento das classes. Se o dataset for muito desbalanceado,
métricas como accuracy podem ser enganosas (um modelo que sempre diz "não cancela"
teria 73% de acerto, mas seria inútil).

In [ ]:
print("=" * 60)
print("DISTRIBUTION OF THE TARGET (CHURN) VARIABLE" )
print("=" * 60)

target_counts = data_frame["Churn"].value_counts()
target_percentages = data_frame["Churn"].value_counts(normalize=True) * 100

print(f"\nTarget Counts")
for idx, count in target_counts.items():
    label = "Remained" if idx == "No" else "Churned"
    print(f" {label}: {count} ({target_percentages[idx]:.1f}%)")

# Visualization of target distribution
fig, axes = mtl.subplots(1, 2, figsize=(14, 5))

colors = ['#2196F3', '#FF5722']
axes[0].bar(['Permaneceu (No)', 'Cancelou (Yes)'], target_counts.values, color=colors)
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição de Churn')
axes[0].grid(axis='y', alpha=0.3)

axes[1].pie(target_counts.values, labels=['Permaneceu', 'Cancelou'],
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Proporção de Churn')

mtl.tight_layout()
mtl.show()

ratio = target_counts.min() / target_counts.max()
print(f"\nChurn Ratio (Minority/Majority): {ratio:.2f}")
print("Unbalanced dataset (~26% churn). We need to use metrics beyond accuracy.")

#### 3.4 Análise de Variáveis Categóricas vs. Churn

Aqui vemos quais features categóricas tem relação com o churn. Isso ajuda a entender quais variáveis podem ser boas preditoras.

In [ ]:
categorical_cols = data_frame.select_dtypes(include="object").columns.drop("customerID", "Churn")

n = len(categorical_cols)
ncols = 3
nrows = int(np.ceil(n / ncols))

for idx, col in enumerate(categorical_cols):
    # Taxa de churn por categoria
    churn_rate = data_frame.groupby(col)['Churn'].apply(lambda x: (x == "Yes").mean() * 100)
    churn_rate = churn_rate.sort_values(ascending=False)

    bars = axes[idx].bar(range(len(churn_rate)), churn_rate.values, color='#FF5722', alpha=0.8)
    axes[idx].set_xticks(range(len(churn_rate)))
    axes[idx].set_xticklabels(churn_rate.index, rotation=45, ha='right', fontsize=8)
    axes[idx].set_ylabel('Taxa de Churn (%)')
    axes[idx].set_title(f'{col}', fontweight='bold')
    axes[idx].set_ylim(0, 60)
    axes[idx].axhline(y=target_pct['Yes'], color='gray', linestyle='--', alpha=0.5, label='Média geral')
    axes[idx].grid(axis='y', alpha=0.3)

for ax in axes[n:]:
    fig.delaxes(ax)

mtl.suptitle('Taxa de Churn por Variável Categórica', fontsize=14, fontweight='bold', y=1.02)
mtl.tight_layout()
mtl.show()

#### 3.5 Análise de Variáveis Numéricas

Distribuições e boxplots das variáveis numéricas, segmentadas por Churn.

In [ ]:
numerical_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = mtl.subplots(1, 3, figzize=(16, 10))

for idx, col in enumerate(numerical_cols):
    for churn_status, color, label in [("No", "#2196F3", "Permaneceu"), ("Yes", "#FF5722", "Cancelou")]:
        axes[0, idx].hist(df[df['Churn'] == churn_val][col], bins=30, alpha=0.6,
                          color=color, label=label, density=True)
    axes[0, idx].set_title(f'Distribuição: {col}', fontweight='bold')
    axes[0, idx].set_xlabel(col)
    axes[0, idx].set_ylabel('Densidade')
    axes[0, idx].legend()
    axes[0, idx].grid(alpha=0.3)

    sns.boxplot(data=df, x='Churn', y=col, ax=axes[1, idx], palette=colors)
    axes[1, idx].set_title(f'Boxplot: {col} por Churn', fontweight='bold')
    axes[1, idx].grid(axis='y', alpha=0.3)

mtl.tight_layout();
mtl.show();

# Estatísticas descritivas por grupo
print("\nEstatísticas por grupo de Churn:")
df.groupby('Churn')[num_cols].describe().round(2)

#### 3.6 Matriz de Correlação

In [ ]:
# Criar versão numérica do target para correlação
df_corr = data_frame.copy()
df_corr['Churn_num'] = (df_corr['Churn'] == 'Yes').astype(int)
df_corr['SeniorCitizen'] = df_corr['SeniorCitizen'].astype(int)

# Correlação das variáveis numéricas com o target
num_for_corr = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_num']
corr_matrix = df_corr[num_for_corr].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1)
mtl.title('Matriz de Correlação', fontsize=14, fontweight='bold')
mtl.tight_layout()
mtl.show()

print("\nCorrelações com Churn (ordenadas):")
print(corr_matrix['Churn_num'].sort_values(ascending=False))

#### 3.7 Análise de Outliers

In [ ]:
print("=" * 60)
print("ANÁLISE DE OUTLIERS (Z-Score > 3)")
print("=" * 60)

for col in numerical_cols:
    z_scores = np.abs(stats.zscore(data_frame))
    outliers = (z_scores > 3).sum()
    print(f"  {col}: {outliers} outliers ({outliers/len(data_frame)*100:.1f}%)")

    print("\nNenhum tratamento agressivo necessário — os valores fazem sentido no contexto de negócio.")


#### 3.8 Data Readiness - Checklist

Verificação final de que os dados estão prontos para modelagem.

In [ ]:
print("=" * 60)
print("DATA READINESS CHECKLIST")
print("=" * 60)

checks = {
    'Volume suficiente (>= 5.000 registros)': len(data_frame) >= 5000,
    'Features suficientes (>= 10)': data_frame.shape[1] >= 10,
    'Variável alvo presente (Churn)': 'Churn' in data_frame.columns,
    'Classificação binária': data_frame['Churn'].nunique() == 2,
    'Sem missing values': data_frame.isnull().sum().sum() == 0,
    'Sem duplicatas': data_frame.duplicated().sum() == 0,
    'Dataset público (sem restrição legal)': True,
}

for check, passed in checks.items():
    status = 'OK' if passed else 'FALHA'
    print(f"  [{status}] {check}")

all_passed = all(checks.values())
print(f"\n{'Todos os checks passaram! Dados prontos.' if all_passed else 'Alguns checks falharam.'}")

#### 4. Pré-processamento

Passos:
1. Remover `customerID` (é um identificador, não uma feature preditiva)
2. Converter o target `Churn` para binário (0/1)
3. Codificar variáveis categóricas (One-Hot Encoding)
4. Separar features (X) e target (y)
5. Split treino/teste com estratificação


In [ ]:
# Copiar para não alterar o original
df_processed = data_frame.copy()

# 1. Remover customerID (identificador, não feature)
df_processed.drop(columns=['customerID'], inplace=True)

# 2. Converter target para binário
df_processed['Churn'] = (df_processed['Churn'] == 'Yes').astype(int)

# 3. One-Hot Encoding das variáveis categóricas
# Em Java seria como criar um Map<String, Integer> para cada categoria.
# get_dummies faz isso automaticamente, criando colunas binárias.
# drop_first=True evita multicolinearidade (coluna redundante).
cat_columns = df_processed.select_dtypes(include='object').columns.tolist()
print(f"Colunas categóricas para encoding: {cat_columns}")

df_encoded = pd.get_dummies(df_processed, columns=cat_columns, drop_first=True)

# Garantir que todas as colunas são numéricas
# (get_dummies pode gerar bool em vez de int)
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f"\nShape após encoding: {df_encoded.shape}")
print(f"Colunas: {df_encoded.columns.tolist()}")

In [ ]:
# 4. Separar features (X) e target (y)

#Aqui você estou criando o conjunto de Features (Atributos). pego o meuy DataFrame e removo apenas a coluna 'Churn'. 
# O que sobra são todas as informações que "explicam" o problema.
X = df_encoded.drop(columns=['Churn'])

#Aqui isolo o meu Target. 
# É a resposta que o modelo quer aprender a prever. 
# No caso, se o cliente cancelou o serviço (Churn) ou não
y = df_encoded['Churn']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nDistribuição do target:")
print(y.value_counts())

# 5 Split treino/teste (80/20) com estratificação
# Estratificação garante que a proporção de churn é mantida em ambos os conjuntos.
# Em Java seria como um StratifiedSampler.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y # Mantém proporção de churn em treino e teste
)

print(f"\nTreino: {X_train.shape[0]} amostras ({y_train.mean()*100:.1f}% churn)")
print(f"Teste:  {X_test.shape[0]} amostras ({y_test.mean()*100:.1f}% churn)")

In [ ]:
# Persistir dados processados
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_encoded.to_csv(PROCESSED_DIR / "telco_churn_processed.csv", index=False)
print(f"Dados processados salvos em: {PROCESSED_DIR / 'telco_churn_processed.csv'}")

### 5. Definição de Métricas

#### Métricas Técnicas

Para churn, **accuracy sozinha é enganosa** porque o dataset é desbalanceado (73% não-churn).
Um modelo que sempre diz "não cancela" teria 73% de accuracy mas 0% de utilidade.

Métricas escolhidas:
- **AUC-ROC**: Capacidade geral de separar churn vs. não-churn (0.5 = aleatório, 1.0 = perfeito)
- **PR-AUC** (Average Precision): Mais informativa em datasets desbalanceados
- **F1-Score**: Equilíbrio entre precision e recall
- **Recall**: Proporção de churners que conseguimos detectar (queremos alto)

#### Métrica de Negócio

**Custo de churn evitado**: Se detectamos um churner e a retenção funciona, economizamos
o valor que esse cliente geraria nos próximos meses. Vamos simular isso após os baselines.

In [ ]:
def evaluate_model(model, X_test, y_test, model_name="Modelo"):
    """Avalia um modelo com as 4+ métricas definidas.
        
    Returns:
        dict com todas as métricas
    """

    # respostas do modelo para o conjunto de teste
    # aqui o modelo da uma resposta direta: Sim, esse cliente vai cancelar = 1 ou Não, ele vai ficar = 0
    y_pred = model.predict(X_test)
    
    # Probabilidades (para AUC-ROC e PR-AUC)
    if hasattr(model, 'predict_proba'):
        #Aqui eh mais sutil. 
        # o modelo ñ diz apenas Sim ou Não, ele diz a probabilidade. 
        # Ex: Tenho 87% de certeza que esse cliente vai cancelar. O [:, 1] serve para pegar apenas a probabilidade da classe positiva 
        #(o cancelamento).
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = y_pred.astype(float)

        #Accuracy (Acuracia): O basico. De 100 clientes, quantos eu acertei no total?
        #Precision (Precisão): dos clientes que eu disse que iam cancelar, quantos realmente cancelaram? (Evita alarmes falsos).
        #Recall (Revocação): De todos os clientes que cancelaram de verdade, quantos eu consegui detectar? (Evita deixar passar cancelamentos).
        #F1-Score: Um equilíbrio entre Precisão e Recall. Ótimo para quando NAo saber qual dos dois é mais importante.
        #ROC-AUC e PR-AUC: Medem quão bem o modelo consegue separar as classes. Quanto mais perto de 1.0, melhor é o modelo.
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
    }
    
    print(f"\n{'=' * 50}")
    print(f"{model_name}")
    print(f"{'=' * 50}")
    for name, value in metrics.items():
        print(f"  {name:>12s}: {value:.4f}")
    #função pronta do Scikit-Learn 
    # que imprime uma tabela detalhando o desempenho para quem Permaneceu e para quem Cancelou separadamente.
    print(f"\n{classification_report(y_test, y_pred, target_names=['Permaneceu', 'Cancelou'])}")
    
    #guarda as metrics para comparar modelos depois (ex: Random Forest vs Regressão Logística)
    #guarda y_pred para criar uma Matriz de Confusão.
    return metrics, y_pred, y_proba

## 6. Baselines + MLflow Tracking

### são baselines?

Baselines são modelos simples que servem como referência. Se seu modelo "inteligente" não
superar o baseline, algo está errado.

**Baselines escolhidos:**
1. **DummyClassifier (stratified)**: Prediz aleatoriamente mantendo a proporção do dataset.
   É o "chute informado" qualquer modelo deve superar isso.
2. **Regressão Logística**: Modelo linear simples, rápido e interpretável.
   É o baseline clássico para classificação binária.

### MLflow Tracking

MLflow é como um "Git para experimentos de ML". Registra parâmetros, métricas e artefatos
de cada experimento, permitindo comparar diferentes runs.

Para visualizar: rode `mlflow ui` no terminal e acesse http://localhost:5000

In [ ]:
# Configurar MLflow
# Em produção, aponta para o servidor MLflow (docker-compose)
# Para desenvolvimento local, usa o diretório mlruns/
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("churn_prediction")

# Versão do dataset (rastreabilidade)
DATASET_VERSION = "v1.0-telco-ibm"
DATASET_SIZE = len(data_frame)

print(f"MLflow experiment: churn_prediction")
print(f"Dataset version: {DATASET_VERSION}")
print(f"Dataset size: {DATASET_SIZE}")

In [ ]:
# ── Baseline 1: DummyClassifier ──────────────────────────────────────────────
# Prediz aleatoriamente mantendo a proporção do dataset.
# Qualquer modelo real DEVE superar isso.
# O Dummy define o desempenho mínimo aceitável.

with mlflow.start_run(run_name="baseline_dummy_stratified"):
    dummy = DummyClassifier(strategy='stratified', random_state=RANDOM_SEED)
    dummy.fit(X_train, y_train)
    
    metrics_dummy, y_pred_dummy, y_proba_dummy = evaluate_model(
        dummy, X_test, y_test, "Baseline — DummyClassifier (stratified)"
    )
    
    # Registrar no MLflow
    mlflow.log_param("model_type", "DummyClassifier")
    mlflow.log_param("strategy", "stratified")
    mlflow.log_param("dataset_version", DATASET_VERSION)
    mlflow.log_param("dataset_size", DATASET_SIZE)
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("random_state", RANDOM_SEED)
    
    for name, value in metrics_dummy.items():
        mlflow.log_metric(name, value)
    
    mlflow.sklearn.log_model(dummy, "model")
    print("\nRegistrado no MLflow.")

In [ ]:
# -- Baseline 2: Regressão Logística -----------------------------------------
# Modelo linear clássico. Rápido, interpretável, bom baseline.

with mlflow.start_run(run_name="baseline_logistic_regression"):
    # Escalar features (Regressão Logística é sensível à escala)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    logisticRegression = LogisticRegression(
        random_state=RANDOM_SEED,
        max_iter=1000,
        class_weight='balanced'  # Compensa o desbalanceamento
    )
    logisticRegression.fit(X_train_scaled, y_train)
    
    metrics_lr, y_pred_lr, y_proba_lr = evaluate_model(
        logisticRegression, X_test_scaled, y_test, "Baseline — Regressão Logística (balanced)"
    )
    
    # Validação cruzada estratificada (requisito do desafio)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    cv_results = cross_validate(
        logisticRegression, X_train_scaled, y_train,
        cv=cv,
        scoring=['accuracy', 'f1', 'roc_auc'],
        return_train_score=True
    )
    
    print(f"\nValidação Cruzada (5-fold):")
    for metric in ['accuracy', 'f1', 'roc_auc']:
        scores = cv_results[f'test_{metric}']
        print(f"  {metric}: {scores.mean():.4f} (+/- {scores.std():.4f})")
    
    # Registrar no MLflow
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("dataset_version", DATASET_VERSION)
    mlflow.log_param("dataset_size", DATASET_SIZE)
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("random_state", RANDOM_SEED)
    mlflow.log_param("scaling", "StandardScaler")
    
    for name, value in metrics_lr.items():
        mlflow.log_metric(name, value)
    
    # Métricas de CV
    for metric in ['accuracy', 'f1', 'roc_auc']:
        mlflow.log_metric(f"cv_{metric}_mean", cv_results[f'test_{metric}'].mean())
        mlflow.log_metric(f"cv_{metric}_std", cv_results[f'test_{metric}'].std())
    
    mlflow.sklearn.log_model(logisticRegression, "model")
    print("\nRegistrado no MLflow.")

### 7. Comparação e Visualização dos Baselines

In [ ]:
# Tabela comparativa
# Construção do DataFrame (pd.DataFrame)
# usando uma técnica de Python chamada Dictionary Unpacking (os dois asteriscos **).
# **metrics_dummy: O seu dicionário de métricas (Acurácia, F1, etc.) é "descompactado". 
# Em vez de ser um dicionário dentro de uma célula, cada métrica vira uma coluna na tabela.
# O resultado é uma tabela onde cada linha é um modelo e cada coluna é uma métrica que você calculou na função evaluate_model
#
results = pd.DataFrame([
    {'Modelo': 'DummyClassifier', **metrics_dummy},
    {'Modelo': 'Logistic Regression', **metrics_lr},
])

print("\n" + "=" * 70)
print("TABELA COMPARATIVA DOS BASELINES")
print("=" * 70)
#to_string(index=False): Remove aquela coluna de números (0, 1, 2) 
# que o Pandas coloca automaticamente A esquerda, deixando a tabela mais limpa.
print(results.to_string(index=False, float_format='{:.4f}'.format))

#Por que isso é importante?
#Sem essa tabela, você teria que ficar subindo e descendo o console para comparar os números. 
# Com ela, você responde visualmente:

#O modelo real é melhor que o aleatório? 
# Se a Logistic Regression não tiver números significativamente maiores que o DummyClassifier, 
# algo está muito errado com seus dados ou com o modelo.
#
# Onde ele é melhor? 
# As vezes, a Regressão Logística ganha em Recall, mas perde em Precisão. 
# A tabela deixa esse trade-off (troca) evidente.
#
#

In [ ]:
# Visualização: métricas lado a lado
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']

fig, ax = mtl.subplots(figsize=(12, 6))
x = np.arange(len(metrics_to_plot))
width = 0.35

bars1 = ax.bar(x - width/2, [metrics_dummy[m] for m in metrics_to_plot], width,
               label='DummyClassifier', color='#90CAF9', edgecolor='black')
bars2 = ax.bar(x + width/2, [metrics_lr[m] for m in metrics_to_plot], width,
               label='Logistic Regression', color='#FF8A65', edgecolor='black')

ax.set_ylabel('Score')
ax.set_title('Comparação de Baselines ->  Métricas', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

# Valores nas barras
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=8)

mtl.tight_layout()
mtl.show()

In [ ]:
# Matriz de Confusão -> Regressão Logística
fig, axes = mtl.subplots(1, 2, figsize=(14, 5))

#O frafico da Esquerda: Matriz de Confusão

# Este comando desenha visualmente o que disse lá pra cima:
# quantos clientes o modelo acertou que permaneceriam, quantos acertou que cancelariam, 
# e os erros (falsos positivos e falsos negativos).

#cmap='Blues': 
# Pinta o gráfico de azul. 
# Quanto mais escuro o azul em um quadrado, mais clientes estão "caídos" naquela categoria.
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr, display_labels=['Permaneceu', 'Cancelou'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Matriz de Confusão -> Logistic Regression')

# Curva ROC
# O Gráfico da Direita: Curva ROC (Receiver Operating Characteristic)
# A Curva ROC avalia a capacidade do modelo de distinguir entre as duas classes em diferentes níveis de "certeza".

# A Linha Tracejada (k--): 
# Representa o modelo aleatório (o DummyClassifier). 
# É uma linha diagonal que cruza o gráfico. Se o modelo estiver em cima dessa linha, ele é inútil.

# A Curva do Modelo:  
# queremos q a linha da Regressão Logística suba rapidamente em direção ao canto superior esquerdo.

RocCurveDisplay.from_predictions(
    y_test, y_proba_lr, name='Logistic Regression', ax=axes[1]
)
# AUC (Área Sob a Curva): É o número que resume esse gráfico
# 0.5: Tão bom quanto jogar uma moeda.
# 1.0: Modelo perfeito.
# 0.80 a 0.90: Um modelo muito bom.
axes[1].plot([0, 1], [0, 1], 'k--', label='Aleatório (AUC=0.5)')
axes[1].set_title('Curva ROC — Logistic Regression')
axes[1].legend()

mtl.tight_layout()
mtl.show()

# Por que olhar os dois juntos?

# Eles se complementam para contar a história completa:

# A Matriz de Confusão da o resultado "aqui e agora" (com base em um corte de 50% de probabilidade, geralmente). 
# Ela diz: "Neste exato momento, erramos 15 clientes".

# A Curva ROC te diz o potencial do modelo. 
# Ela mostra: "Se a gente for um pouco mais rigoroso ou mais flexível na nossa regra de decisão, o modelo consegue melhorar?".

## 8. Análise de Custo (Trade-off FP vs FN)

### Métrica de Negócio: Custo de Churn Evitado

Simulação simples:
- **Custo de perder um cliente (Falso Negativo):** MonthlyCharges x 12 meses (LTV aproximado)
- **Custo de retenção desnecessária (Falso Positivo):** 20% do MonthlyCharges (desconto/oferta)

In [ ]:
# Análise de custo com Regressão Logística
avg_monthly = data_frame['MonthlyCharges'].mean()

CUSTO_FN = avg_monthly * 12   # Perder o cliente: 12 meses de receita
CUSTO_FP = avg_monthly * 0.20  # Retenção desnecessária: 20% de desconto

cm = confusion_matrix(y_test, y_pred_lr)
tn, fp, fn, tp = cm.ravel()

custo_total_fn = fn * CUSTO_FN
custo_total_fp = fp * CUSTO_FP
receita_salva = tp * CUSTO_FN  # Clientes que detectamos e retemos

print("=" * 60)
print("ANÁLISE DE CUSTO — Regressão Logística")
print("=" * 60)
print(f"\nPremissas:")
print(f"  MonthlyCharges médio: ${avg_monthly:.2f}")
print(f"  Custo de perder cliente (FN): ${CUSTO_FN:.2f} (12 meses de receita)")
print(f"  Custo de retenção desnecessária (FP): ${CUSTO_FP:.2f} (20% desconto)")
print(f"\nMatriz de Confusão:")
print(f"  True Negatives  (acertou 'fica'):    {tn}")
print(f"  False Positives (errou → 'cancela'): {fp}")
print(f"  False Negatives (errou → 'fica'):    {fn}")
print(f"  True Positives  (acertou 'cancela'): {tp}")
print(f"\nImpacto Financeiro:")
print(f"  Receita em risco (FN): ${custo_total_fn:,.2f}")
print(f"  Custo retenção desnec. (FP): ${custo_total_fp:,.2f}")
print(f"  Receita potencialmente salva (TP): ${receita_salva:,.2f}")
print(f"  Saldo líquido (salva - FP): ${receita_salva - custo_total_fp:,.2f}")

## 9. Conclusões da Etapa 1

### Achados da EDA
- Dataset com 7.043 registros e 20+ features após encoding
- Target desbalanceado (26% churn)
- TotalCharges tinha missing values (11 clientes com tenure=0)
- Variáveis mais correlacionadas com churn: tipo de contrato, tenure, serviços de streaming

### Baselines
- DummyClassifier: baseline mínimo, 50% em F1
- Regressão Logística: já mostra resultados razoáveis, serve como referência sólida

### Próximos passos (Etapa 2)
- Construir MLP em PyTorch
- Feature engineering avançado
- Comparar MLP vs. baselines usando >= 4 métricas
- Otimizar hiperparâmetros

In [ ]:
print("Etapa 1 concluída!")
print("\nPara visualizar os experimentos no MLflow:")
print("  cd ml-engineering-churn-prediction")
print("  mlflow ui --backend-store-uri mlruns")
print("  Abra: http://localhost:5000")